# Predicting Student Health Risk - Baseline Modeling

This is the **public baseline** notebook for S6E7. It now keeps three important lessons in one place:

1. **Raw local accuracy can be misleading**: `hgb_unweighted` had strong CV accuracy but scored only `0.85038` publicly.
2. **Class balance matters**: `hgb_balanced` remains the public champion at `0.90603`.
3. **Next improvement direction**: add **CatBoost balanced** with row-safe lifestyle interaction features, while keeping one **notebook-generated** `submission.csv`.

The notebook compares candidates, selects a champion using **balanced accuracy** as the public-score proxy, and writes a single `submission.csv` from that champion.

## 1. Setup

The notebook runs on Kaggle and locally. The input resolver scans common competition mount paths and falls back to `/kaggle/input/**/train.csv` when necessary.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.utils.class_weight import compute_sample_weight

SEED = 42
N_SPLITS = 5
EPS = 1e-6

candidate_dirs = [
    Path('/kaggle/input/playground-series-s6e7'),
    Path('/kaggle/input/predicting-student-health-risk'),
    Path('../data'),
    Path('data'),
]
DATA_DIR = next(
    (path for path in candidate_dirs if (path / 'train.csv').exists()),
    None,
)
if DATA_DIR is None and Path('/kaggle/input').exists():
    for train_path in Path('/kaggle/input').glob('**/train.csv'):
        parent = train_path.parent
        if (parent / 'test.csv').exists() and (parent / 'sample_submission.csv').exists():
            DATA_DIR = parent
            break
if DATA_DIR is None:
    raise FileNotFoundError('Could not locate train/test/sample_submission CSV files.')

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('..')
PRED_DIR = OUTPUT_DIR / 'predictions'
PRED_DIR.mkdir(exist_ok=True)
print(f'Using data directory: {DATA_DIR}')

## 2. Load Data

`health_condition` is the target. The target is heavily imbalanced, so every candidate must be judged by **minority recall**, **balanced accuracy**, and **prediction mix**, not raw accuracy alone.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

id_col = sample_submission.columns[0]
target_candidates = [col for col in train.columns if col not in test.columns]
target = target_candidates[0]

base_features = [col for col in test.columns if col in train.columns and col != id_col]
y = train[target]
classes = np.array(sorted(y.unique()))
target_share = y.value_counts(normalize=True).reindex(classes)

print(f'train shape: {train.shape}')
print(f'test shape: {test.shape}')
print(f'target: {target}')
display(y.value_counts().to_frame('count'))
display(target_share.to_frame('target_share'))

## 3. Row-Safe Feature Engineering

EDA showed strong class separation from **sleep**, **stress**, **activity**, **BMI**, and lifestyle categories. The engineered features below are row-safe and available for both train and test.

The goal is not feature volume; it is to expose obvious lifestyle interactions for tree models.

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add row-safe lifestyle interaction features."""
    out = df.copy()
    feature_cols = [col for col in base_features if col in out.columns]
    out['missing_count'] = out[feature_cols].isna().sum(axis=1)

    out['activity_intensity'] = out['step_count'] / (out['exercise_duration'] + 1.0)
    out['calorie_per_step'] = out['calorie_expenditure'] / (out['step_count'] + 1.0)
    out['exercise_per_sleep'] = out['exercise_duration'] / (out['sleep_duration'] + 1.0)
    out['steps_per_sleep'] = out['step_count'] / (out['sleep_duration'] + 1.0)
    out['bmi_sleep_interaction'] = out['bmi'] * out['sleep_duration']
    out['activity_sleep_score'] = (
        out['sleep_duration'].fillna(out['sleep_duration'].median())
        + np.log1p(out['step_count'].fillna(0.0)) / 2.0
        + out['exercise_duration'].fillna(0.0) / 30.0
    )

    out['low_sleep_flag'] = (out['sleep_duration'] < 6).astype(int)
    out['high_sleep_flag'] = (out['sleep_duration'] >= 8).astype(int)
    out['high_stress_low_sleep'] = (
        (out['stress_level'] == 'high') & (out['sleep_duration'] < 6)
    ).astype(int)
    out['poor_sleep_high_stress'] = (
        (out['sleep_quality'] == 'poor') & (out['stress_level'] == 'high')
    ).astype(int)
    out['active_good_sleep'] = (
        (out['physical_activity_level'] == 'active')
        & (out['sleep_quality'] == 'good')
    ).astype(int)
    out['low_stress_active'] = (
        (out['stress_level'] == 'low')
        & (out['physical_activity_level'] == 'active')
    ).astype(int)
    out['smoking_high_stress'] = (
        (out['smoking_alcohol'] == 'yes') & (out['stress_level'] == 'high')
    ).astype(int)

    return out

train_fe = add_features(train)
test_fe = add_features(test)

feature_cols = [col for col in test_fe.columns if col in train_fe.columns and col != id_col and col != target]
cat_cols = train_fe[feature_cols].select_dtypes(include='object').columns.tolist()
num_cols = [col for col in feature_cols if col not in cat_cols]

print(f'features: {len(feature_cols)}')
print(f'numeric features: {len(num_cols)}')
print(f'categorical features: {cat_cols}')

## 4. Validation And Candidate Strategy

We use **5-fold stratified CV**. Candidate selection now uses **balanced accuracy** because public leaderboard behavior favored class-balanced predictions over raw local accuracy.

Candidates:

- `hgb_balanced_fe`: balanced HGB with engineered features;
- `catboost_balanced_fe`: CatBoost with native categorical handling and engineered features.

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
X = train_fe[feature_cols]
X_test = test_fe[feature_cols]


def summarize_predictions(name: str, oof_pred: pd.Series, test_pred: pd.Series) -> dict:
    """Build metric and prediction-mix summary for one candidate."""
    row = {
        'candidate': name,
        'accuracy': accuracy_score(y, oof_pred),
        'balanced_accuracy': balanced_accuracy_score(y, oof_pred),
        'macro_f1': f1_score(y, oof_pred, average='macro'),
        'weighted_f1': f1_score(y, oof_pred, average='weighted'),
    }
    oof_share = oof_pred.value_counts(normalize=True).reindex(classes).fillna(0.0)
    test_share = test_pred.value_counts(normalize=True).reindex(classes).fillna(0.0)
    for cls in classes:
        row[f'oof_{cls}_share'] = oof_share.loc[cls]
        row[f'test_{cls}_share'] = test_share.loc[cls]
    return row

## 5. Candidate 1: Balanced HGB With Engineered Features

This keeps the current public champion's class-balanced behavior but adds the row-safe engineered features. It is the baseline to beat.

In [ ]:
def make_hgb_pipeline() -> Pipeline:
    """Create a balanced HGB pipeline using engineered features."""
    preprocess = ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median', add_indicator=True), num_cols),
            (
                'cat',
                Pipeline(
                    [
                        ('imputer', SimpleImputer(strategy='most_frequent')),
                        (
                            'encoder',
                            OrdinalEncoder(
                                handle_unknown='use_encoded_value',
                                unknown_value=-1,
                            ),
                        ),
                    ]
                ),
                cat_cols,
            ),
        ],
        remainder='drop',
    )
    model = HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=160,
        l2_regularization=0.05,
        early_stopping=True,
        random_state=SEED,
    )
    return Pipeline([('preprocess', preprocess), ('model', model)])

hgb_oof = pd.Series(index=train.index, dtype=object)
hgb_test_votes = []

for fold, (trn_idx, val_idx) in enumerate(cv.split(X, y), start=1):
    pipeline = make_hgb_pipeline()
    X_trn, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_trn, y_val = y.iloc[trn_idx], y.iloc[val_idx]
    sample_weight = compute_sample_weight('balanced', y_trn)
    pipeline.fit(X_trn, y_trn, model__sample_weight=sample_weight)
    hgb_oof.iloc[val_idx] = pipeline.predict(X_val)
    hgb_test_votes.append(pipeline.predict(X_test))
    print(f'HGB fold {fold} complete')

hgb_vote_frame = pd.DataFrame(np.column_stack(hgb_test_votes))
hgb_test_pred = hgb_vote_frame.mode(axis=1)[0]
print('Balanced HGB complete')

## 6. Candidate 2: Balanced CatBoost With Native Categoricals

CatBoost is a natural next step because it handles categorical fields directly and can learn interactions such as **stress × sleep**, **activity × exercise**, and **sleep quality × lifestyle**.

In [ ]:
def prepare_catboost_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Fill categorical missing values for CatBoost while preserving numeric NaNs."""
    out = df.copy()
    for col in cat_cols:
        out[col] = out[col].astype('object').fillna('missing').astype(str)
    return out

X_cat = prepare_catboost_frame(X)
X_test_cat = prepare_catboost_frame(X_test)
cat_feature_indices = [X_cat.columns.get_loc(col) for col in cat_cols]

cat_oof = pd.Series(index=train.index, dtype=object)
cat_test_votes = []

for fold, (trn_idx, val_idx) in enumerate(cv.split(X_cat, y), start=1):
    X_trn, X_val = X_cat.iloc[trn_idx], X_cat.iloc[val_idx]
    y_trn, y_val = y.iloc[trn_idx], y.iloc[val_idx]

    train_pool = Pool(X_trn, y_trn, cat_features=cat_feature_indices)
    val_pool = Pool(X_val, y_val, cat_features=cat_feature_indices)
    test_pool = Pool(X_test_cat, cat_features=cat_feature_indices)

    model = CatBoostClassifier(
        loss_function='MultiClass',
        eval_metric='TotalF1',
        auto_class_weights='Balanced',
        iterations=450,
        learning_rate=0.06,
        depth=6,
        l2_leaf_reg=6.0,
        random_seed=SEED + fold,
        od_type='Iter',
        od_wait=50,
        verbose=False,
        allow_writing_files=False,
    )
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    cat_oof.iloc[val_idx] = model.predict(val_pool).reshape(-1)
    cat_test_votes.append(model.predict(test_pool).reshape(-1))
    print(f'CatBoost fold {fold} complete; best_iteration={model.get_best_iteration()}')

cat_vote_frame = pd.DataFrame(np.column_stack(cat_test_votes))
cat_test_pred = cat_vote_frame.mode(axis=1)[0]
print('Balanced CatBoost complete')

## 7. Compare Candidates And Select Champion

The **champion** is selected by **balanced accuracy**, with **macro F1** and **prediction mix** as supporting diagnostics. This mirrors what the public leaderboard taught us from the failed unweighted submission.

In [ ]:
results = [
    summarize_predictions('hgb_balanced_fe', hgb_oof, hgb_test_pred),
    summarize_predictions('catboost_balanced_fe', cat_oof, cat_test_pred),
]
comparison = pd.DataFrame(results).set_index('candidate').sort_values(
    ['balanced_accuracy', 'macro_f1'], ascending=False
)
display(comparison)

champion_name = comparison.index[0]
if champion_name == 'hgb_balanced_fe':
    champion_oof = hgb_oof
    champion_test_pred = hgb_test_pred
else:
    champion_oof = cat_oof
    champion_test_pred = cat_test_pred

print(f'Selected champion by balanced accuracy: {champion_name}')

## 8. Champion Diagnostics

We inspect per-class precision/recall and confusion matrix before considering a leaderboard submission.

In [ ]:
print(classification_report(y, champion_oof, digits=4))

cm = pd.DataFrame(
    confusion_matrix(y, champion_oof, labels=classes),
    index=[f'true_{cls}' for cls in classes],
    columns=[f'pred_{cls}' for cls in classes],
)
display(cm)

pd.DataFrame(
    {
        id_col: train[id_col] if id_col in train else train.index,
        'oof_pred': champion_oof,
        'target': y,
    }
).to_csv(PRED_DIR / f'{champion_name}_oof_predictions.csv', index=False)
comparison.to_csv(OUTPUT_DIR / 'baseline_model_comparison.csv')

## 9. Create Notebook-Generated Submission

The notebook writes exactly one `submission.csv` from the selected **champion**. This keeps the public notebook as the source of truth.

In [ ]:
submission = sample_submission.copy()
submission.iloc[:, 1] = champion_test_pred.values
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)

display(submission.head())
display(submission.iloc[:, 1].value_counts(normalize=True).to_frame('prediction_share'))
print(f'Wrote submission to {OUTPUT_DIR / "submission.csv"}')
print(f'Champion candidate: {champion_name}')

## 10. Submission Gate

Submit only if diagnostics preserve the class-balanced behavior that produced the current champion score of `0.90603`.

**Good signs**

- balanced accuracy improves over `hgb_balanced`;
- macro F1 improves without collapsing minority recall;
- `fit` and `unhealthy` prediction shares remain meaningfully represented;
- the notebook-generated artifact is reproducible.

**Bad signs**

- the model becomes too close to the rejected unweighted prediction mix;
- minority recall collapses;
- local accuracy improves while balanced behavior weakens.